# Walkthrough of how to run SIRF inside XNAT

This tutorial shows how to 
 - create a docker image which can run SIRF image reconstruction inside XNAT
 - install the XNAT plugins for MR and PET raw data in XNAT
 - set-up the docker image inside XNAT
 - upload MR and PET raw data to XNAT
 - reconstruct the uploaded data inside XNAT

In [ ]:
import xnat4tests
from pathlib import Path
import os
import subprocess
import time
import stir
import zenodo_get
import zipfile

from src.main.mrd_2_xnat import mrd_2_xnat
from src.main.listmode_2_xnat import listmode_2_xnat

### 1. Settings
Here we define some versions and the location where the plugins can be found. 

In [ ]:
xnat_version = "1.9.2"
xnat_container_service_version = "3.7.2"

mrd_plugin_link = Path(
    "https://github.com/SyneRBI/xnat-mrd/releases/download/v1.0.0/mrd-plugin-1.0.0.jar"
)
interfile_plugin_link = Path(
    "https://github.com/SyneRBI/xnat-interfile/releases/download/v1.0.0/interfile-plugin-1.0.0.jar"
)

### 2. SIRF Docker image

As a first step we are going to create a docker image with the reconstruction code inside. 
We use the latest SIRF docker image for this: ghcr.io/synerbi/sirf:petric2

The easiest to do this is open a terminal and go to `XNAT-SIRF/docker`.

Then run `docker build . -t sirf_for_xnat`. 

This will download the SIRF docker image and create a new image `sirf_for_xnat`. 
If you want to modify the reconstruction scripts, have a look at `XNAT-SIRF/docker/reco_scripts/mr_direct_recon.py` 
and `XNAT-SIRF/docker/reco_scripts/pet_osem_recon.py`.

Once the docker image is successfully build we can continue with setting up the xnat4tests instance.

### 3. XNAT 4 TESTS
We install xnat4tests as an example of how to interact with XNAT.

In [ ]:
# Create folder for xnat4tests
os.makedirs(Path(os.getcwd()) / ".xnat4tests", exist_ok=True)
xnat_root_dir = Path(os.getcwd()) / ".xnat4tests/root"
docker_build_dir = Path(os.getcwd()) / ".xnat4tests/build"

# Settings for XNAT test server
xnat_config = xnat4tests.Config(
    xnat_root_dir=xnat_root_dir,
    docker_build_dir=docker_build_dir,
    docker_image="xnat_sirf_xnat4tests",
    docker_container="xnat_sirf_xnat4tests",
    build_args={
        "xnat_version": xnat_version,
        "xnat_cs_plugin_version": xnat_container_service_version,
    },
)
xnat4tests.start_xnat(xnat_config)
connection = xnat4tests.connect(xnat_config)

### 4. DOWNLOAD AND INSTALL PLUGINS

In [ ]:
!curl -L -O {str(mrd_plugin_link)}
!curl -L -O {str(interfile_plugin_link)}

In [ ]:
# Check which plugins are installed
plugin_dir = Path("/data/xnat/home/plugins")
status = subprocess.run(
    [
        "docker",
        "exec",
        "xnat_sirf_xnat4tests",
        "ls",
        plugin_dir.as_posix(),
    ],
    check=True,
    capture_output=True,
    text=True,
)
plugins_list = status.stdout.split("\n")

# Install MRD and INTERFILE plugins if not already installed
for plugin_link in (mrd_plugin_link, interfile_plugin_link):
    if plugin_link.name not in plugins_list:
        try:
            subprocess.run(
                [
                    "docker",
                    "cp",
                    str(plugin_link.name),
                    f"xnat_sirf_xnat4tests:{(plugin_dir / plugin_link.name).as_posix()}",
                ],
                check=True,
            )
        except subprocess.CalledProcessError as e:
            raise RuntimeError(
                f"Command {e.cmd} returned with error code {e.returncode}: {e.output}"
            ) from e

xnat4tests.restart_xnat(xnat_config)
time.sleep(30)  # Wait for XNAT to restart

You can now go to http://localhost:8080 in a browser and login with admin/admin. 
If you go to `Administer` and then `Data types` you will see the MRD and INTERFILE data types.

### 5. Setup SIRF reconstruction in XNAT

In [ ]:
!curl -u admin:admin \
  -X POST "http://localhost:8080/xapi/commands" \
  -H "Content-Type: application/json" \
  -d @mr_manifest.json 

!curl -u admin:admin \
  -X POST "http://localhost:8080/xapi/commands" \
  -H "Content-Type: application/json" \
  -d @pet_manifest.json

### 6. Create project and enable SIRF reconstruction for it

In [ ]:
# Create Project
project_id = "SIRF"
xnat_session = xnat4tests.connect(xnat_config)
response = xnat_session.put(f"/data/archive/projects/{project_id}")
xnat_session.projects.clearcache()

Now we have to enable the commands globally and for the project. Follow the following steps:
 - go to http://localhost:8080 in a browser and login with admin/admin
 - select `Administer` and then `Plugin settings`
 - click on `Command Configurations` and enable the MR and PET image reconstruction
 - select `Browse` -> `My Projects` -> `SIRF`
 - click `Project Settings` and enable the MR and PET image reconstruction

### 7. Download MR and PET data from zenodo

In [ ]:
raw_data_folder = Path(os.getcwd()) / ".zenodo_data"
os.makedirs(raw_data_folder, exist_ok=True)

# Download MR raw data
zenodo_get.download(record="2633785", retry_attempts=5, output_dir=raw_data_folder)
with zipfile.ZipFile(
    raw_data_folder / Path("PTB_ACRPhantom_GRAPPA.zip"), "r"
) as zip_data:
    zip_data.extractall(raw_data_folder)

# Download PET raw data
zenodo_get.download(record="1304454", retry_attempts=5, output_dir=raw_data_folder)
with zipfile.ZipFile(raw_data_folder / Path("NEMA_IQ.zip"), "r") as zip_data:
    zip_data.extractall(raw_data_folder)

### 8. Upload MR and PET data

In [ ]:
# Settings
subject_id = "Patient"
mr_session_id = "MrSession"
mr_scan_id = "MrScan"
mrd_file_path = raw_data_folder / Path(
    "PTB_ACRPhantom_GRAPPA/ptb_resolutionphantom_fully_ismrmrd.h5"
)
pet_session_id = "PetSession"
pet_scan_id = "PetScan"
pet_path = raw_data_folder / Path("NEMA_IQ/")

# Create Subject
project = xnat_session.projects[project_id]
subject = xnat_session.classes.SubjectData(label=subject_id, parent=project)

# MR
xnat_hdr = mrd_2_xnat(mrd_file_path)
mr_session = xnat_session.classes.MrSessionData(label=mr_session_id, parent=subject)
response = xnat_session.put(f"{mr_session.uri}/scans/{mr_scan_id}", query=xnat_hdr)
mr_session.clearcache()
scan = mr_session.scans[mr_scan_id]
scan_resource = scan.create_resource("MR_RAW")
scan_resource.upload(mrd_file_path, mrd_file_path.name)

# PET
interfile_lm_path = sorted(pet_path.glob("*.l.hdr"))[0]
header = stir.ListModeData.read_from_file(str(interfile_lm_path))
xnat_hdr = listmode_2_xnat(header)
pet_session = xnat_session.classes.PetSessionData(label=pet_session_id, parent=subject)
response = xnat_session.put(f"{pet_session.uri}/scans/{pet_scan_id}", query=xnat_hdr)
pet_session.clearcache()
scan = pet_session.scans[pet_scan_id]
scan_resource = scan.create_resource("PET_RAW")
for interfile_file_path in pet_path.iterdir():
    if interfile_file_path.is_file():
        scan_resource.upload(interfile_file_path, interfile_file_path.name)

### 9. SIRF reconstruction in XNAT

Now we can reconstruct the data:
 - go to http://localhost:8080 in a browser and login with admin/admin
 - select `Browse` -> `My Projects` -> `SIRF`
 - select subject `Patient` -> `MR session`
 - select `MrScan` and via `Run Container` select `mr_recon` 
 - once the reconstruction has run through, the image data can be accessed through `Manage Files` -> `DICOM`  
 - you can repeat the same for `PET session` -> `PetScan` -> `Run Container` -> `pet_recon` 

### Clean-up

In [ ]:
for project in xnat_session.projects:
    for subject in project.subjects.values():
        xnat_session.delete(
            path=f"/data/projects/{project.id}/subjects/{subject.label}",
            query={"removeFiles": "True"},
        )
    project.subjects.clearcache()
xnat_session.disconnect()